# Stats File Analysis Report

Analyzes per-node `stats-<peer_id>.json` files bundled in result tarballs.
Covers: run inventory, message correctness, late messages, arrival lag, byte volumes, committee consistency, peer drops.
Final cell prints a self-contained markdown report.

In [1]:
# Cell 0 — Imports & Config
import json
import re
import tarfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tabulate import tabulate

RESULTS_DIR = Path("../results")

# Protocol display names (strip session suffix for readability)
PROTO_SHORT = {
    "exante": "exante",
    "expost": "expost",
    "exanteMDAG": "exanteMDAG",
    "expostMDAG": "expostMDAG",
}

def ns_to_ms(ns):
    return ns / 1_000_000

def bytes_to_mb(b):
    return b / (1024 * 1024)

def parse_sid(sid):
    """Extract n_nodes and rep from sid like 'sweep-500n-rep-01-20260622-210000'."""
    m = re.match(r"sweep-(\d+)n-rep-(\d+)-(.+)", sid)
    if m:
        return int(m.group(1)), int(m.group(2)), m.group(3)
    return None, None, sid

def short_proto_id(proto_id, sid):
    """Strip session suffix from protocol IDs like '/exante/1.0.0/<sid>'."""
    return proto_id.replace(f"/{sid}", "").replace("/1.0.0", "").replace("/mdag", "mdag")

print(f"Results directory: {RESULTS_DIR.resolve()}")
tarballs = sorted(RESULTS_DIR.glob("*.tar.gz"))
print(f"Found {len(tarballs)} tarballs")

Results directory: /Users/mamorski/repos/committee-analysis/results
Found 18 tarballs


In [2]:
# Cell 1 — Load All Stats
# Returns all_stats: list of dicts, one per node per run
# Each dict = raw JSON data + parsed run params

def load_all_stats(results_dir):
    all_stats = []
    for tgz in sorted(results_dir.glob("*.tar.gz")):
        with tarfile.open(tgz, "r:gz") as tf:
            stat_members = [
                m for m in tf.getmembers()
                if "/stats/" in m.name and m.name.endswith(".json")
            ]
            for sm in stat_members:
                try:
                    data = json.load(tf.extractfile(sm))
                except Exception as e:
                    print(f"PARSE ERROR {sm.name}: {e}")
                    continue
                sid = data.get("sid", tgz.stem.replace(".tar", ""))
                n_nodes, rep, ts = parse_sid(sid)
                data["_sid"] = sid
                data["_n_nodes"] = n_nodes
                data["_rep"] = rep
                data["_ts"] = ts
                data["_tarball"] = tgz.name
                all_stats.append(data)
    return all_stats

all_stats = load_all_stats(RESULTS_DIR)
print(f"Loaded {len(all_stats)} node stats across {len(tarballs)} runs")

Loaded 9900 node stats across 18 runs


In [3]:
# Cell 2 — Run Inventory Table

# Count node logs and stats per tarball
tarball_info = {}
for tgz in sorted(RESULTS_DIR.glob("*.tar.gz")):
    with tarfile.open(tgz, "r:gz") as tf:
        members = tf.getmembers()
        node_logs = sum(1 for m in members if "/node-" in m.name)
        stat_files = sum(1 for m in members if "/stats/" in m.name and m.name.endswith(".json"))
    sid = tgz.stem.replace(".tar", "")
    n_nodes, rep, ts = parse_sid(sid)
    tarball_info[sid] = {
        "sid": sid,
        "n_nodes": n_nodes,
        "rep": rep,
        "timestamp": ts,
        "node_logs": node_logs,
        "stats_files": stat_files,
        "match": "OK" if node_logs == stat_files else f"MISMATCH({node_logs} logs vs {stat_files} stats)",
    }

inventory_rows = list(tarball_info.values())
inventory_df = pd.DataFrame(inventory_rows)
print(tabulate(inventory_df, headers="keys", tablefmt="pipe", showindex=False))

| sid                               |   n_nodes |   rep | timestamp       |   node_logs |   stats_files | match   |
|:----------------------------------|----------:|------:|:----------------|------------:|--------------:|:--------|
| sweep-500n-rep-01-20260622-210000 |       500 |     1 | 20260622-210000 |         500 |           500 | OK      |
| sweep-500n-rep-02-20260622-212615 |       500 |     2 | 20260622-212615 |         500 |           500 | OK      |
| sweep-500n-rep-03-20260622-215230 |       500 |     3 | 20260622-215230 |         500 |           500 | OK      |
| sweep-520n-rep-01-20260622-221846 |       520 |     1 | 20260622-221846 |         520 |           520 | OK      |
| sweep-520n-rep-02-20260622-224502 |       520 |     2 | 20260622-224502 |         520 |           520 | OK      |
| sweep-520n-rep-03-20260622-231118 |       520 |     3 | 20260622-231118 |         520 |           520 | OK      |
| sweep-540n-rep-01-20260622-233734 |       540 |     1 | 20260622-23373

In [4]:
# Cell 3 — Stats File Correctness
# Check total_messages == valid_messages for all nodes/protocols

correctness = defaultdict(lambda: {"nodes_with_invalid": 0, "total_invalid_msgs": 0})

for s in all_stats:
    sid = s["_sid"]
    for proto_name, proto in s.get("protocols", {}).items():
        total = proto.get("total_messages", [])
        valid = proto.get("valid_messages", [])
        invalid = sum(t - v for t, v in zip(total, valid) if t > v)
        if invalid > 0:
            key = (sid, proto_name)
            correctness[key]["nodes_with_invalid"] += 1
            correctness[key]["total_invalid_msgs"] += invalid

if correctness:
    rows = [
        {"sid": k[0], "protocol": k[1], **v}
        for k, v in sorted(correctness.items())
    ]
    correctness_df = pd.DataFrame(rows)
    print("INVALID MESSAGES DETECTED:")
    print(tabulate(correctness_df, headers="keys", tablefmt="pipe", showindex=False))
else:
    print("All nodes: total_messages == valid_messages across all protocols. No invalid messages.")

All nodes: total_messages == valid_messages across all protocols. No invalid messages.


In [5]:
# Cell 4 — Late Messages

late_data = defaultdict(lambda: {"nodes_with_late": 0, "total_late_msgs": 0, "max_lateness_rounds": 0})

for s in all_stats:
    sid = s["_sid"]
    for proto_name, proto in s.get("protocols", {}).items():
        late = proto.get("late_messages", 0)
        max_late = proto.get("max_lateness", 0)
        if late > 0:
            key = (sid, proto_name)
            late_data[key]["nodes_with_late"] += 1
            late_data[key]["total_late_msgs"] += late
            late_data[key]["max_lateness_rounds"] = max(
                late_data[key]["max_lateness_rounds"], max_late
            )

if late_data:
    rows = [
        {"sid": k[0], "protocol": k[1], **v}
        for k, v in sorted(late_data.items())
    ]
    late_df = pd.DataFrame(rows)
    print("LATE MESSAGES DETECTED:")
    print(tabulate(late_df, headers="keys", tablefmt="pipe", showindex=False))
else:
    print("No late messages in any run.")

LATE MESSAGES DETECTED:
| sid                               | protocol   |   nodes_with_late |   total_late_msgs |   max_lateness_rounds |
|:----------------------------------|:-----------|------------------:|------------------:|----------------------:|
| sweep-600n-rep-01-20260623-033401 | exante     |               600 |             21812 |                     1 |
| sweep-600n-rep-01-20260623-033401 | expost     |               600 |             21721 |                     1 |


In [6]:
# Cell 5 — Arrival Lag Analysis
# Aggregate arrival_lag per run per protocol.
# Per node: compute mean of mean_ns across rounds, max of max_ns across rounds.
# Per run/protocol: median of per-node means, p95 of per-node maxes, global max.

# Collect per-node per-run per-protocol lag vectors
lag_vectors = defaultdict(lambda: {"node_means": [], "node_maxes": []})

for s in all_stats:
    sid = s["_sid"]
    n_nodes = s["_n_nodes"]
    for proto_name, proto in s.get("protocols", {}).items():
        lags = proto.get("arrival_lag", [])
        if not lags:
            continue
        node_mean = np.mean([e["mean_ns"] for e in lags])
        node_max = max(e["max_ns"] for e in lags)
        key = (sid, n_nodes, proto_name)
        lag_vectors[key]["node_means"].append(node_mean)
        lag_vectors[key]["node_maxes"].append(node_max)

lag_rows = []
for (sid, n_nodes, proto_name), v in sorted(lag_vectors.items()):
    means = v["node_means"]
    maxes = v["node_maxes"]
    lag_rows.append({
        "sid": sid,
        "n_nodes": n_nodes,
        "protocol": proto_name,
        "median_mean_ms": round(ns_to_ms(np.median(means)), 1),
        "p95_mean_ms": round(ns_to_ms(np.percentile(means, 95)), 1),
        "p95_max_ms": round(ns_to_ms(np.percentile(maxes, 95)), 1),
        "global_max_ms": round(ns_to_ms(max(maxes)), 1),
    })

lag_df = pd.DataFrame(lag_rows)

# Print exante/expost separately from MDAG (very different scales)
for proto_group, label in [
    (["exante", "expost"], "exante / expost"),
    (["exanteMDAG", "expostMDAG"], "exanteMDAG / expostMDAG"),
]:
    sub = lag_df[lag_df["protocol"].isin(proto_group)].copy()
    if not sub.empty:
        print(f"\n### Lag — {label}")
        print(tabulate(sub, headers="keys", tablefmt="pipe", showindex=False))


### Lag — exante / expost
| sid                               |   n_nodes | protocol   |   median_mean_ms |   p95_mean_ms |   p95_max_ms |   global_max_ms |
|:----------------------------------|----------:|:-----------|-----------------:|--------------:|-------------:|----------------:|
| sweep-500n-rep-01-20260622-210000 |       500 | exante     |           5949.5 |        6159.5 |      28839.2 |         28856.7 |
| sweep-500n-rep-01-20260622-210000 |       500 | expost     |           5943.9 |        6143.1 |      28841.9 |         28861.9 |
| sweep-500n-rep-02-20260622-212615 |       500 | exante     |           6424.2 |        6645.6 |      30795.1 |         30817   |
| sweep-500n-rep-02-20260622-212615 |       500 | expost     |           6409.9 |        6651   |      30802   |         30822.3 |
| sweep-500n-rep-03-20260622-215230 |       500 | exante     |           4655   |        4806.4 |      22269.3 |         22289.7 |
| sweep-500n-rep-03-20260622-215230 |       500 | expost

In [7]:
# Cell 6 — Byte Volume Analysis

# Per-run summary (from bytes.summary per node)
byte_summary_data = defaultdict(lambda: {
    "total_in": [], "total_out": [],
    "app_in": [], "app_out": [],
    "overhead_in": [], "overhead_out": [],
    "n_nodes": 0,
})

for s in all_stats:
    sid = s["_sid"]
    n_nodes = s["_n_nodes"]
    bsummary = s.get("bytes", {}).get("summary", {})
    if not bsummary:
        continue
    d = byte_summary_data[sid]
    d["n_nodes"] = n_nodes
    d["total_in"].append(bsummary.get("total_in", 0))
    d["total_out"].append(bsummary.get("total_out", 0))
    d["app_in"].append(bsummary.get("app_in", 0))
    d["app_out"].append(bsummary.get("app_out", 0))
    d["overhead_in"].append(bsummary.get("overhead_in", 0))
    d["overhead_out"].append(bsummary.get("overhead_out", 0))

bytes_rows = []
for sid, d in sorted(byte_summary_data.items()):
    med_total_in = np.median(d["total_in"])
    med_total_out = np.median(d["total_out"])
    med_app_in = np.median(d["app_in"])
    med_overhead_in = np.median(d["overhead_in"])
    app_ratio = (med_app_in / med_total_in * 100) if med_total_in > 0 else 0
    overhead_ratio = (med_overhead_in / med_total_in * 100) if med_total_in > 0 else 0
    bytes_rows.append({
        "sid": sid,
        "n_nodes": d["n_nodes"],
        "median_total_in_MB": round(bytes_to_mb(med_total_in), 2),
        "median_total_out_MB": round(bytes_to_mb(med_total_out), 2),
        "app_pct": round(app_ratio, 1),
        "overhead_pct": round(overhead_ratio, 1),
    })

bytes_df = pd.DataFrame(bytes_rows)
print("### Byte Volume Summary (per-node medians)")
print(tabulate(bytes_df, headers="keys", tablefmt="pipe", showindex=False))

# Per-protocol byte breakdown (aggregate median in/out across nodes, per run)
proto_byte_data = defaultdict(lambda: defaultdict(lambda: {"in": [], "out": []}))

for s in all_stats:
    sid = s["_sid"]
    by_proto = s.get("bytes", {}).get("by_protocol", {})
    for proto_id, bdata in by_proto.items():
        short = short_proto_id(proto_id, sid)
        proto_byte_data[sid][short]["in"].append(bdata.get("in", 0))
        proto_byte_data[sid][short]["out"].append(bdata.get("out", 0))

proto_byte_rows = []
for sid, protos in sorted(proto_byte_data.items()):
    n_nodes, rep, ts = parse_sid(sid)
    for proto_short, vals in sorted(protos.items()):
        if not vals["in"]:
            continue
        proto_byte_rows.append({
            "sid": sid,
            "protocol": proto_short,
            "median_in_MB": round(bytes_to_mb(np.median(vals["in"])), 3),
            "median_out_MB": round(bytes_to_mb(np.median(vals["out"])), 3),
        })

proto_bytes_df = pd.DataFrame(proto_byte_rows)
# Filter to app-level protocols only (skip libp2p internals)
app_protos = proto_bytes_df[proto_bytes_df["protocol"].str.contains("exante|expost|mdag", case=False)].copy()
print("\n### Byte Volume by Protocol (per-node medians, app protocols only)")
print(tabulate(app_protos, headers="keys", tablefmt="pipe", showindex=False))

### Byte Volume Summary (per-node medians)
| sid                               |   n_nodes |   median_total_in_MB |   median_total_out_MB |   app_pct |   overhead_pct |
|:----------------------------------|----------:|---------------------:|----------------------:|----------:|---------------:|
| sweep-500n-rep-01-20260622-210000 |       500 |                13.87 |                 12.97 |      91.2 |            8.8 |
| sweep-500n-rep-02-20260622-212615 |       500 |                14.99 |                 14.03 |      91.7 |            8.3 |
| sweep-500n-rep-03-20260622-215230 |       500 |                11.38 |                 10.41 |      89.7 |           10.3 |
| sweep-520n-rep-01-20260622-221846 |       520 |                14.81 |                 13.86 |      91.5 |            8.5 |
| sweep-520n-rep-02-20260622-224502 |       520 |                14.49 |                 13.55 |      91.3 |            8.7 |
| sweep-520n-rep-03-20260622-231118 |       520 |                14.11 |   

In [8]:
# Cell 7 — Committee Analysis

# Collect committee sets per node per run
run_committees = defaultdict(dict)  # sid -> {node_id: frozenset of member IDs}

for s in all_stats:
    sid = s["_sid"]
    node_id = s.get("node_id", "?")
    committee = s.get("committee", [])
    member_ids = frozenset(m["id"] for m in committee)
    run_committees[sid][node_id] = member_ids

# Grade distribution (should be all grade 5 = honest)
all_grades = defaultdict(int)
for s in all_stats:
    for m in s.get("committee", []):
        all_grades[m.get("grade", "?")] += 1

committee_rows = []
for sid, node_comms in sorted(run_committees.items()):
    n_nodes, rep, ts = parse_sid(sid)
    committees = list(node_comms.values())
    first = committees[0] if committees else frozenset()
    consistent = all(c == first for c in committees)
    sizes = [len(c) for c in committees]
    committee_rows.append({
        "sid": sid,
        "n_nodes": n_nodes,
        "rep": rep,
        "committee_size": sizes[0] if sizes else 0,
        "consistent": "YES" if consistent else f"NO ({sum(1 for c in committees if c != first)} disagree)",
    })

committee_df = pd.DataFrame(committee_rows)
print("### Committee Consistency")
print(tabulate(committee_df, headers="keys", tablefmt="pipe", showindex=False))

# Pivot: committee size by n_nodes × rep
pivot = committee_df.pivot_table(
    index="n_nodes", columns="rep", values="committee_size", aggfunc="first"
)
pivot.columns = [f"rep{int(c)}" for c in pivot.columns]
pivot["mean"] = pivot.mean(axis=1).round(1)
pivot = pivot.reset_index()
print("\n### Committee Size by Network Size")
print(tabulate(pivot, headers="keys", tablefmt="pipe", showindex=False))

# Grade distribution
print(f"\nGrade distribution across all committee memberships: {dict(all_grades)}")

### Committee Consistency
| sid                               |   n_nodes |   rep |   committee_size | consistent   |
|:----------------------------------|----------:|------:|-----------------:|:-------------|
| sweep-500n-rep-01-20260622-210000 |       500 |     1 |               51 | YES          |
| sweep-500n-rep-02-20260622-212615 |       500 |     2 |               56 | YES          |
| sweep-500n-rep-03-20260622-215230 |       500 |     3 |               41 | YES          |
| sweep-520n-rep-01-20260622-221846 |       520 |     1 |               55 | YES          |
| sweep-520n-rep-02-20260622-224502 |       520 |     2 |               54 | YES          |
| sweep-520n-rep-03-20260622-231118 |       520 |     3 |               52 | YES          |
| sweep-540n-rep-01-20260622-233734 |       540 |     1 |               41 | YES          |
| sweep-540n-rep-02-20260623-000350 |       540 |     2 |               47 | YES          |
| sweep-540n-rep-03-20260623-003006 |       540 |     

In [9]:
# Cell 8 — Peer Drops

peer_drop_rows = []
for s in all_stats:
    drops = s.get("peer_drops", [])
    if drops:
        peer_drop_rows.append({
            "sid": s["_sid"],
            "node_id": s.get("node_id", "?")[:20],
            "drop_count": len(drops),
            "sample": str(drops[0])[:80],
        })

if peer_drop_rows:
    peer_drops_df = pd.DataFrame(peer_drop_rows)
    print("PEER DROPS DETECTED:")
    print(tabulate(peer_drops_df, headers="keys", tablefmt="pipe", showindex=False))
else:
    print("No peer drops recorded in any run.")

No peer drops recorded in any run.


In [ ]:
# Cell 8b — Network Graph Analysis
# Built from the `neighbors` field in each node's stats file.
import networkx as nx

per_run_graphs = defaultdict(nx.Graph)
for s in all_stats:
    sid = s["_sid"]
    node_id = s.get("node_id", "?")
    per_run_graphs[sid].add_node(node_id)
    for nb_peer in s.get("neighbors", []):
        per_run_graphs[sid].add_edge(node_id, nb_peer)

graph_rows = []
for sid, g in sorted(per_run_graphs.items()):
    n_nodes_cfg, rep, ts = parse_sid(sid)
    degrees = [d for _, d in g.degree()]
    comp_sizes = sorted([len(c) for c in nx.connected_components(g)], reverse=True)
    graph_rows.append({
        "sid": sid,
        "n_nodes": n_nodes_cfg,
        "graph_nodes": g.number_of_nodes(),
        "edges": g.number_of_edges(),
        "deg_min": int(np.min(degrees)),
        "deg_max": int(np.max(degrees)),
        "deg_mean": round(float(np.mean(degrees)), 1),
        "deg_p95": round(float(np.percentile(degrees, 95)), 1),
        "components": len(comp_sizes),
        "largest_comp": comp_sizes[0] if comp_sizes else 0,
    })

graph_df = pd.DataFrame(graph_rows)
print("### Network Graph (from neighbors field)")
print(tabulate(graph_df, headers="keys", tablefmt="pipe", showindex=False))

In [ ]:
# Cell 9 — Markdown Report → written to file

from IPython.display import Markdown, display

sections = []

# Header
sections.append("# Stats File Analysis Report\n")
sections.append(
    f"Runs analysed: **{len(tarballs)}** | "
    f"Total node stats: **{len(all_stats)}**\n"
)

# 1. Run Inventory
sections.append("## 1. Run Inventory\n")
sections.append(tabulate(inventory_df, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")

# 2. Stats Correctness
sections.append("## 2. Stats File Correctness\n")
if correctness:
    rows = [{"sid": k[0], "protocol": k[1], **v} for k, v in sorted(correctness.items())]
    sections.append("**INVALID MESSAGES DETECTED:**\n")
    sections.append(tabulate(pd.DataFrame(rows), headers="keys", tablefmt="pipe", showindex=False))
else:
    sections.append("`total_messages == valid_messages` across all protocols. **No invalid messages.**")
sections.append("")

# 3. Late Messages
sections.append("## 3. Late Messages\n")
if late_data:
    rows = [{"sid": k[0], "protocol": k[1], **v} for k, v in sorted(late_data.items())]
    sections.append(tabulate(pd.DataFrame(rows), headers="keys", tablefmt="pipe", showindex=False))
else:
    sections.append("No late messages in any run.")
sections.append("")

# 4. Arrival Lag
sections.append("## 4. Arrival Lag\n")
for proto_group, label in [
    (["exante", "expost"], "exante / expost"),
    (["exanteMDAG", "expostMDAG"], "exanteMDAG / expostMDAG"),
]:
    sub = lag_df[lag_df["protocol"].isin(proto_group)].copy()
    if not sub.empty:
        sections.append(f"### {label}\n")
        sections.append(tabulate(sub, headers="keys", tablefmt="pipe", showindex=False))
        sections.append("")

# 5. Byte Volumes
sections.append("## 5. Byte Volumes\n")
sections.append("### Per-Node Medians (Summary)\n")
sections.append(tabulate(bytes_df, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")
sections.append("### Per-Protocol Medians (App Protocols)\n")
sections.append(tabulate(app_protos, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")

# 6. Network Graph
sections.append("## 6. Network Graph\n")
sections.append(tabulate(graph_df, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")

# 7. Committee
sections.append("## 7. Committee Analysis\n")
sections.append("### Consistency per Run\n")
sections.append(tabulate(committee_df, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")
sections.append("### Committee Size by Network Size\n")
sections.append(tabulate(pivot, headers="keys", tablefmt="pipe", showindex=False))
sections.append("")
sections.append(f"**Grade distribution:** {dict(all_grades)}")
sections.append("")

# 8. Peer Drops
sections.append("## 8. Peer Drops\n")
if peer_drop_rows:
    sections.append(tabulate(pd.DataFrame(peer_drop_rows), headers="keys", tablefmt="pipe", showindex=False))
else:
    sections.append("No peer drops recorded in any run.")
sections.append("")

report_md = "\n".join(sections)

# Write to file
report_path = RESULTS_DIR / "stats_report.md"
report_path.write_text(report_md)
print(f"Report written to: {report_path.resolve()}")
print(f"({len(report_md)} chars, {report_md.count(chr(10))} lines)")

# Also display inline
display(Markdown(report_md))